In [ ]:
#Instalación de los SDK oficiales necesarios para la ejecución
!python -m pip install google-genai openai

In [1]:
import os

#Definición del escenario
caso_estudio = {
    "expediente_id": "EXP-2026-9941-MUN",
    "nomenclatura": "Circunscripción II, Sección C, Manzana 54, Parcela 18",
    "dimensiones_lote": "12 metros de frente por 30 metros de fondo (Superficie total: 360 m²)",
    "proyecto_solicitado": (
        "Construcción de un edificio residencial de 4 plantas (Altura total: 12 metros), "
        "con una ocupación en planta baja de 235 m² y sin retiros laterales"
    ),
    "codigo_urbano_aplicable": (
        "Zonificación: R2 (Residencial de Media Densidad)\n"
        "Factor de Ocupación del Suelo (FOS) máximo permitido: 0.60 (Equivale a un máximo de 216 m² construibles en planta)\n"
        "Altura máxima permitida: 9 metros (Equivalente a 3 plantas)\n"
        "Retiro obligatorio de frente: 3 metros. Retiro obligatorio de fondo: 4 metros"
    )
}


In [2]:
def construir_prompt(caso: dict) -> tuple:
    """Construye las instrucciones del sistema y del usuario basadas en el caso."""
    system_instruction = (
        "Actúas como un agrimensor, inspector municipal y asesor legal experto en desarrollo urbano. "
        "Tu objetivo es procesar datos catastrales de forma ultra rápida, analítica y estructurada. "
        "Sé estricto con los datos provistos: evalúa únicamente las infracciones del proyecto solicitado frente a la norma aplicable."
    )
    
    user_prompt = f"""
Analiza la viabilidad técnica y legal del siguiente expediente catastral:

[DATOS TÉCNICOS DEL EXPEDIENTE]
- Identificador: {caso['expediente_id']}
- Dimensiones del Lote: {caso['dimensiones_lote']}
- Propuesta de Obra Solicitada: {caso['proyecto_solicitado']}

[MARCO LEGAL DE REFERENCIA - CATASTRO GENERAL]
{caso['codigo_urbano_aplicable']}

[FORMATO DE SALIDA MANDATORIO]
1. RESUMEN EJECUTIVO PARA EL CIUDADANO (Debes listar explícitamente TODAS las infracciones detectadas: Exceso de FOS, exceso de altura y violaciones de retiros).
2. INFORME TÉCNICO DE RESTRICCIONES (BENCHMARK)
3. DIRECTIVA VISUAL AVANZADA (PROMPT DE IMAGEN)
"""
    return system_instruction, user_prompt


In [3]:
def consultar_modelo_openai(client, system_prompt: str, user_prompt: str):
    """Realiza la llamada real al SDK de OpenAI controlando tokens."""
    try:
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.0,
            max_tokens=1000  #Límite estricto exigido
        )
        return completion
    except Exception as e:
        print(f"❌ Error al conectar con el SDK de OpenAI: {e}")
        return None

In [4]:
def parsear_respuesta_openai(completion):
    """Extrae el contenido del reporte técnico y calcula las métricas de consumo real."""
    if completion:
        print("\n" + "="*80)
        print("REPORTES URBANÍSTICOS Y VISUALES GENERADOS POR EL SCI")
        print("="*80)
        print(completion.choices[0].message.content)
        print("="*80)
        
        #Medición de uso mediante response.usage
        print("\n📊 MÉTRICAS DE CONSUMO (OpenAI response.usage):")
        print(f"Tokens de Entrada (Prompt): {completion.usage.prompt_tokens}")
        print(f"Tokens de Salida (Completion): {completion.usage.completion_tokens}")
        print(f"Tokens Totales: {completion.usage.total_tokens}")
        
        # Cálculo de costos reales basado en tarifas oficiales de gpt-4o-mini
        costo_usd = (completion.usage.prompt_tokens * 0.15 / 1_000_000) + (completion.usage.completion_tokens * 0.60 / 1_000_000)
        print(f"Costo real de la consulta: ${costo_usd:.6f} USD")

In [ ]:

#EJECUCIÓN CON GOOGLE GEMINI SDK

import os
from google import genai
from google.genai import types

#Copiá acá tu API Key de Google AI Studio
MI_CLAVE_REAL = "Poner Clave propia" 

if not MI_CLAVE_REAL:
    print("⚠️ Por favor, introduce tu GEMINI_API_KEY real en la variable MI_CLAVE_REAL.")
else:
    #Inicialización directa pasándole la clave del usuario
    client_gemini = genai.Client(api_key=MI_CLAVE_REAL)
    sys_prompt, user_prompt = construir_prompt(caso_estudio)
    
    print("Conectando con el modelo gemini-3.8-flash ...\n")
    try:
        response = client_gemini.models.generate_content(
            model='gemini-3.8-flash',
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=sys_prompt,
                temperature=0.0,
                max_output_tokens=1000
            )
        )
        
        print("="*80)
        print(response.text)
        print("="*80)
        
        print("\n📊 MÉTRICAS DE CONSUMO (Gemini Usage):")
        print(f"Tokens de Entrada (Prompt): {response.usage_metadata.prompt_token_count}")
        print(f"Tokens de Salida (Candidates): {response.usage_metadata.candidates_token_count}")
        print(f"Tokens Totales: {response.usage_metadata.total_token_count}")
    except Exception as e:
        print(f"❌ Error en la llamada: {e}")


Conectando con el modelo gemini-3.8-flash ...

)**
        *   Tabla comparativa: Parámetro | Norma R2 | Propuesta | Desvío / Estado Legal.
        *   Surface total: 360 m²

📊 MÉTRICAS DE CONSUMO (Gemini Usage):
Tokens de Entrada (Prompt): 397
Tokens de Salida (Candidates): 40
Tokens Totales: 1393


In [ ]:
import os
from openai import OpenAI

api_key_openai = os.getenv("OPENAI_API_KEY")

if not api_key_openai:
    print("⚠️ Celda configurada para el evaluador. Se requiere la variable 'OPENAI_API_KEY' en el entorno para ejecutar.")
else:
    # Inicialización del cliente oficial de OpenAI
    client_openai = OpenAI(api_key=api_key_openai)
    
    # Orquestación del flujo usando la separación estricta de funciones exigida
    sys_prompt, user_prompt = construir_prompt(caso_estudio)
    completion_real = consultar_modelo_openai(client_openai, sys_prompt, user_prompt)
    parsear_respuesta_openai(completion_real)
